# Creating the grid dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Objectdetection/data")
SEED = 42
SPLITS = ["train", "val", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def class_from_filename(path):
    match = re.match(r"OBJ(\d+)", path.stem.upper())
    return f"OBJ{int(match.group(1)):03d}" if match else None


EXPECTED_CLASSES = sorted({
    class_from_filename(path)
    for split in SPLITS
    for path in (DATA_ROOT / "images" / split).iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
} - {None})

print(f"Classes found: {len(EXPECTED_CLASSES)}")

for split in SPLITS:
    missing = sorted(
        p.name for p in (DATA_ROOT / "images" / split).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
        and not (DATA_ROOT / "labels" / split / f"{p.stem}.txt").exists()
    )
    print(f"{split}: {len(missing)} images without labels {missing[:5]}")

In [ ]:
import random
import yaml
import shutil
from PIL import Image

GRID_DIR = DATA_ROOT / "grid_dataset"
shutil.rmtree(GRID_DIR, ignore_errors=True)
IMAGE_SIZE = 224
GRID_LAYOUTS = [(2, 2), (3, 3), (4, 4), (5, 4), (5, 5)]

random.seed(SEED)


def load_yolo_labels(label_path):
    if not label_path.exists():
        raise FileNotFoundError(f"Missing label file: {label_path}")

    boxes = []
    for line in label_path.read_text().splitlines():
        values = line.split()
        if not values:
            continue
        if len(values) != 5:
            raise ValueError(f"Invalid YOLO label in {label_path}: {line}")
        boxes.append((int(values[0]), *map(float, values[1:])))
    return boxes


def create_grid(split, rows, cols):
    label_dir = DATA_ROOT / "labels" / split
    out_image_dir = GRID_DIR / "images" / split
    out_label_dir = GRID_DIR / "labels" / split
    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    sources = sorted(
        p for p in (DATA_ROOT / "images" / split).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    grid_size = rows * cols
    if len(sources) < grid_size:
        print(f"Skipping {split} {rows}x{cols}: not enough images")
        return

    random.shuffle(sources)
    grid_count = len(sources) // grid_size
    width, height = cols * IMAGE_SIZE, rows * IMAGE_SIZE

    for grid_index in range(grid_count):
        canvas = Image.new("RGB", (width, height))
        labels = []

        batch = sources[grid_index * grid_size:(grid_index + 1) * grid_size]
        for index, image_path in enumerate(batch):
            x0 = (index % cols) * IMAGE_SIZE
            y0 = (index // cols) * IMAGE_SIZE

            with Image.open(image_path) as img:
                if img.size != (IMAGE_SIZE, IMAGE_SIZE):
                    raise ValueError(f"Unexpected image size: {image_path} {img.size}")
                canvas.paste(img.convert("RGB"), (x0, y0))

            for class_id, xc, yc, bw, bh in load_yolo_labels(label_dir / f"{image_path.stem}.txt"):
                labels.append(
                    f"{class_id} {(x0 + xc * IMAGE_SIZE) / width:.6f} "
                    f"{(y0 + yc * IMAGE_SIZE) / height:.6f} "
                    f"{bw * IMAGE_SIZE / width:.6f} {bh * IMAGE_SIZE / height:.6f}"
                )

        name = f"grid_{rows}x{cols}_{grid_index:05d}"
        canvas.save(out_image_dir / f"{name}.jpg", quality=95)
        (out_label_dir / f"{name}.txt").write_text("\n".join(labels))

    print(f"{split}: {rows}x{cols}, generated {grid_count} grids ({width}x{height})")


for split in SPLITS:
    for rows, cols in GRID_LAYOUTS:
        create_grid(split, rows, cols)

## Verify labels and create data.yaml

In [ ]:
class_ids = set()

for split in SPLITS:
    image_dir = GRID_DIR / "images" / split
    label_dir = GRID_DIR / "labels" / split

    images = [p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    missing = [p.name for p in images if not (label_dir / f"{p.stem}.txt").exists()]

    print(f"{split}: {len(images)} images, {len(missing)} missing labels")
    if missing:
        print("Examples:", missing[:5])

    for label_path in label_dir.glob("*.txt"):
        class_ids.update(
            int(line.split()[0])
            for line in label_path.read_text().splitlines()
            if line.strip()
        )

if not class_ids:
    raise ValueError("No object labels found. Check your label files.")

if class_ids != set(range(max(class_ids) + 1)):
    raise ValueError(f"Class IDs must be contiguous from 0. Found: {sorted(class_ids)}")

yaml_path = GRID_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(
        {
            "path": str(GRID_DIR),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "names": EXPECTED_CLASSES,
        },
        f,
        sort_keys=False,
    )

print(f"Number of classes: {len(EXPECTED_CLASSES)}")
print(f"Class 0: {EXPECTED_CLASSES[0]}")
print(f"Class {len(EXPECTED_CLASSES) - 1}: {EXPECTED_CLASSES[-1]}")
print(f"Saved: {yaml_path}")

train: 2628 images, 0 missing labels
val: 560 images, 0 missing labels
test: 561 images, 0 missing labels
Number of classes: 73
Class 0: OBJ001
Class 72: OBJ073
Saved: /content/drive/MyDrive/Objectdetection/data/grid_dataset/data.yaml


# Training YOLOv8

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from ultralytics import YOLO

GRID_DIR = Path("/content/drive/MyDrive/Objectdetection/data/grid_dataset")
RUNS_DIR = Path("/content/drive/MyDrive/Objectdetection/runs")
RUN_NAME = "yolov8s_grid"

model = YOLO("yolov8s.pt")

results = model.train(
    data=str(GRID_DIR / "data.yaml"),
    epochs=100,
    imgsz=1024,
    batch=5,
    patience=20,
    seed=42,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=5, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Objectdetection/data/grid_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0

## Resume training

In [ ]:
model = YOLO(str(RUNS_DIR / RUN_NAME / "weights" / "last.pt"))
model.train(resume=True)

WARNING ⚠️ model '/content/drive/MyDrive/Objectdetection/runs/yolov8s_grid/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x785f6a086660>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

## Evaluate on test set

In [ ]:
best_model = YOLO(str(RUNS_DIR / RUN_NAME / "weights" / "best.pt"))

metrics = best_model.val(
    data=str(GRID_DIR / "data.yaml"),
    split="test",
    imgsz=1024,
    batch=8,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,153,835 parameters, 0 gradients, 28.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.2 ms, read: 0.5±0.2 MB/s, size: 208.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/.shortcut-targets-by-id/1NXn7oX20Aw8ERXLlPt4_IGABaGaJsOWN/Objectdetection/data/grid_dataset/labels/test... 561 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 561/561 2.9it/s 3:16
val: New cache created: /content/drive/.shortcut-targets-by-id/1NXn7oX20Aw8ERXLlPt4_IGABaGaJsOWN/Objectdetection/data/grid_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 2.9it/s 24.4s
                   all        561       7058      0.866      0.783      0.806      0.765
                OBJ0